PROGETTO 1: REALIZZAZIONE DI UNA PIPELINE IBRIDA OCR E NLP

l'OCR e NLP sono due lavori diversi.
OCR: che testo vedo nell'immagine, percezione del testo
NLP: che cosa significa quel testo e quali informazioni mi servono?, interpretazione del testo

Passiamo dal mondo fatto i pixel disordinati a un mondo di dati strutturati. E' il ponte tra la computer visione ed il NLP.

*** OCR ***

L'OCR non significa, testo perfetto, può per esempio, confondere la O per uno zero, o viceversa, a quel punto  dopo OCR serve fare un controllo
Questo controllo può essere fatto in diversi modi: regex, lookup database, regoli di dominio, NER, LLM

- Acquisizione del documento: PDF, JPG, foto, scansione, ecc
- Preprocessing visivo : Computer vision per grayscale, riduzione rumore, thresholding, deskew, ocr
- OCR: trasforma l'immagine in testo
- NLP: trasformo il testo OCR in informazioni utili. Puoi usare diverse tecniche: - regex + keyword - spaCy NER - BERT fine-tuned - LLM

Estrazione del Testo da immagini: Il Motore Tesseract
Trasformare pixel in stringhe codificate.
L'Optical Character Recognition (OCR) è il processo di conversione da immagini di testo digitato, scritto a mano o stampato in testo codificato comprensibile alla macchina.
E' come un traduttore simultaneo che parla due lingue diverse, il linguaggio dei pixel e quallo dei caratteri.
Tesseract è un motore OCR open-source che integra reti neurali basate su layer LSTM (memoria a lungo termine) per riconoscere sequenze di caratteri con alta precisione in oltre cento lingue.

Meccanismo di Funzionamento
Dobbiamo regolare Tesseract come se stessimo regolando la messa a fuoco di un binocolo.
* Page Segmentation Mode (PSM) configura il modo in cui Tesseract analizza il layout dell'immagine, distinguendo tra blocchi di testo, singole righe o parole isolate.
* Engine Mode (OEM) definisce l'algoritmo di riconoscimenti, privilegiando i modelli basati su LSTM per una maggior resilienza al rumore visivo.
* Binarizzazione:  processo di conversione dell'immagine in banco e nero fondamentale per isolare i glifi dallo sfondo, facilitando l'estrazione dei contorni dei caratteri
* DPI e Scaling: importanza della risoluzione dell'immagine originale, idealmente superiore a 300 punti per pollice, per minimizzare l'ambiguità tra caratteri simili.

Spesso però l'immagine che riceviamo è sporca, come la puliamo?

Pre-processing dell'Immagine per l'OCR
- Rdizione del rumore: applicazione di filtri mediano o gaussiani per rimuovere artefatti digitali che potrebbero essere scambiati per punteggiatura
- Correzione dell'Inclinazione (Deskewing): allineamento del testo per garantire che il motore LSTM legga la sequenza di caratteri lungo l'asse corretto.
- Normalizzazione del Contrasto: utilizzo di tecniche come l'istogramma adattivo per far emergere il testo in immagini con illuminazione non uniforme.

Una volta estratto il testo, dobbiamo chiederci, la macchina è sicura di ciò che ha letto?

Analisi della Confidenza
Probabilità a livello di parola
Tesseract restituisce un valore di confidenza per ogni parola estratta. Questo parametro è fondamentale per decidere se accettareil dato o richiederne una revisione.
La confidenzaa media del documento C viene calcolato pesando i valori c_i in ogni token rispetto alla lunghezza totale della sequenza.
Se leggiamo "fattura" con una confidenza di 99 possiamo procedere, alternativamente se la confidenza scenda al 40% la macchina ha visto qualcosa di strano ed ha cercato di dare un senso.
La confidenza media serve per decidere se passare alla fase successiva o se chiedere all'utente di rifare la foto.
Questo è il nostro primo filtro di qualità

Post-processing: Pulizia e Correzione
Curare le ferite dell'estrazione ottica.
L'OCR non è mai perfetto: errori di sostituzione (come lo 0 scambiato per O, o ! in  I) o di segmentazione sono comuni, specialmente in documenti degradati.
In questa fase dobbiamo implementare una pipeline di pulizia che combina espressioni regolari e modelli linguistici per restituire coerenza al testo.

Ma quali sono i ferri del mestiere per questa operazione?

Tecniche di Raffinamento
Correzione sistematica
- Espressioni regolari (Regex): utilissime per correggere pattern prevedibili come date, codici fiscali, o indirizzi mail danneggiati. Sappiamo che in una certa posizione deve esserci una data, possiamo forzare il formato anche se OCR ha scambiato una barra per un numero.
- Dizionario di Dominio: confronto della parole estratte con vocabolario specifico (es. termini medici o legali) per correggere refusi OCR. Se sto leggendo un testo medico, la parola aspirina è più probabile di aspirin@
- Controllo del Contesto: utilizzo di n-grammi per verificare se una sequenza di parole ha senso grammaticalmente o se contiene termini 'impossibili'
- Rimozione di Artefatti: eliminazione automatica di caratteri speciali isolati generati da sporcizia sul sensore dello scanner, tradotto erroneamente in testo.

Ma come facciamo a correggere una parola con non conosciamo?

Correzione Basata sulla Similarità
- Distanza di Modifica: Se una parola non è nel dizionario, non la cancelliamo, cerchiamo il termine più vicino applicando un numero limitato di sostituzioni di caratteri.
- Heuristic Mapping: Definizione di regole di rimpiazzo manuali per i 'falsi amici' visivi, come rimpiazzare '|' con 'I' o 'l' in base alla posizione
- Validazione degli Checksum: per dati numerici sensibili come gli IBAN, applichiamo algoritmi matematici di controllo per confermare l'accuratezza dell'OCR

Ma come misuriamo la distanza tra due parole scritte diversamente=

Metodologia di Confronto Stringhe
L'algoritmo di Levenshtein
La distanza di Levenshtein quantifica la differenza tra due stringhe calcolando il numero minimo di operazioni di inserimento, cancellazione o sostituzione necessarie. Misura quanto sforzo serve per trasformare una stringa in un'altra.
Utiilzziamo questa metrica per mappare gli errori dell'OCR sulla parola più probabile del dizionario di riferimento.

Ora che il testo è pulito e lucido è il momento di estrarne il valore

Analisi Semantica del Testa Estratto
Dalla stringa al significato profondo, l'analisi semantica
Una volta ottenuto e pulito il testo, il passo finale consiste nel processarlo tramite modelli NLP avanzati per estrarne valore decisionale.
Non ci interessano più i pixel ma il significato.
Utiliziamo i Transformer per compiti di classificazione, analisi del sentiment o Named Entity Recognition (NER) sui dati estratti dalle immagini.
E' qui che la pipeline diventa intelligente.

Ma quali sono i compiti che può svolgere questa integrazione?

Integrazione OCR-NLP
L'integrazione tra OCR e NLP ci apre porte incredibili, possiamo fare information extracion e trovare l'importo di una fattura o la data di scadenza di un contratto tra migliaia di righe di testo. 
Wordkflow end-to-end
- Classificazione dei Documenti: identificare automaticamente se l'immagine caricata è una fattura, un contratto o un documento d'identità
- Information Extraction: isolare entità chiave come date di scadenza, importo monetari, o nomi di aziende tramite modelli BERT fine-tuned
- Sentiment Analysis: analizzare il tono di comunicazione cartacee scannerizzate (es. lettere di reclamo) per prioritarizzarne la gestione
- Embeddings di Documento: convertire l'intero testo estratto in un vettore denso per la ricerca semantica in grandi archivi digitali. Trasformiamo l'intera pagina scannerizzata in un unico punto nello spazio semantico, questo ci permette di ricercare documenti non per nome file ma per significato.

Ma come facciamo a rendere la nostra pipeline ancora più robusto e efficiente?
dobbiamo crare un feedback loop
Gestione del Fallimento OCR: Se il nostro modello NLP ci dice che il testo non ha alcun senso semantico, probabilmente l'ocr ha fallito e può richiedere una nuova scansione ocr con parametri PSM diversi.
Feedback Loop: i risultati dell'analisi semantica possono essere usati per migliorare il dizionario di post-processing correggere errori OCR ricorrenti
Efficienza Computazionale: dobbiamo anche essere efficienti, mentra la GPU analizza l imagine, il processore può già iniziare a pulire i primo blocchi di testo estratti.

Ma come confrontiamo due documenti estratti per vedere se sono simili?

Similarità Coseno post-OCR
Confronto vettoriale del testo estratto
Per confrontare il testo estratto con documenti noti, calcoliamo la similarità tra i rispettivi word embeddings prodotti da un Transformer
Il valore di similarità ci indica quanto la semantica del testo scannerizzato sia vicina ai pattern di riferimento del sistema.
Non cercano la parola esatto ma cercano il concetto.

In [3]:
# python -m pip install torchvision --index-url https://download.pytorch.org/whl/cpu
# python -m pip install easyocr
import os
import cv2  # OpenCV: Usata qui per creare un'immagine sintetica (dummy) se non ne esiste una.
import numpy as np  # NumPy: Fondamentale per la gestione di array e matrici (le immagini sono matrici di pixel).
import re  # Regular Expressions: Per la ricerca e sostituzione avanzata di pattern nel testo.
from difflib import get_close_matches  # Utile per correggere parole con errori di battitura (Fuzzy Matching).
import fitz

# ==============================================================================
# 1. CONFIGURAZIONE AMBIENTE E BACKEND
# ==============================================================================
# In alcuni contesti ibridi (es. se usassimo anche Keras), è cruciale definire
# il backend PRIMA di importare librerie di Deep Learning. Qui forziamo l'uso
# di PyTorch per coerenza, evitando che Keras cerchi TensorFlow se installato.
os.environ["KERAS_BACKEND"] = "torch"

import torch      # PyTorch: Il motore di calcolo tensororiale sottostante.
import easyocr    # EasyOCR: Libreria OCR pronta all'uso basata su PyTorch.
from transformers import pipeline  # Hugging Face Transformers: API di alto livello per usare modelli NLP.

# ==============================================================================
# 2. INIZIALIZZAZIONE DEL MOTORE OCR (Optical Character Recognition)
# ==============================================================================
# Questa fase è pesante: carichiamo i pesi della rete neurale (CRAFT per il testo, 
# CRNN per il riconoscimento) nella memoria (RAM o VRAM della GPU).
# ------------------------------------------------------------------------------
print("\n[STEP 0]: Inizializzazione EasyOCR...")

# Classe Reader: è il "cervello" che processa le immagini.
# - ['it', 'en']: Specifichiamo le lingue attese (Italiano e Inglese) per migliorare la precisione.
# - gpu=...: Se CUDA è disponibile, usiamo la scheda video per velocizzare di 10x-50x l'inferenza.
reader = easyocr.Reader(['it', 'en', 'fr'], gpu=torch.cuda.is_available())

# ==============================================================================
# 3. DEFINIZIONE DELLE FUNZIONI (LA PIPELINE)
# ==============================================================================
"""
def extract_text_simplified(image_path):
    
    Fase 1: VISIONE.
    Prende il percorso di un'immagine e usa la rete neurale per estrarre tutto il testo visibile.
   
    print("\n[1] Estrazione testo (OCR in corso)...")
    
    # reader.readtext: Metodo principale di EasyOCR.
    # - image_path: Percorso del file.
    # - detail=0: Semplifica l'output. Invece di darci anche le coordinate (bounding box)
    #   e la confidenza per ogni parola, ci restituisce solo una lista di stringhe trovate.
    results = reader.readtext(image_path, detail=0)
    
    # Uniamo le singole stringhe trovate (es. ["FATTURA", "N.", "2026"]) in un unico blocco di testo.
    return " ".join(results)
"""

def extract_text_simplified(file_path):
    """
    Fase 1: OCR.

    Gestisce sia:
    - immagini JPG/PNG
    - documenti PDF

    Se riceve un PDF, converte ogni pagina in un'immagine e poi applica EasyOCR.
    """

    print("\n[1] Estrazione testo (OCR in corso)...")
    extension = os.path.splitext(file_path)[1].lower()

    # ==========================================================
    # CASO 1: PDF
    # ==========================================================
    if extension == ".pdf":
        print("[INFO] Documento PDF rilevato.")
        document = fitz.open(file_path)
        testo_pagine = []
        for numero_pagina, page in enumerate(document):
            print(f"[OCR] Elaborazione pagina {numero_pagina + 1}/{len(document)}...")

            # Renderizzo la pagina ad una risoluzione maggiore.
            # 2x migliora generalmente la leggibilità per l'OCR.
            matrix = fitz.Matrix(2, 2)

            pix = page.get_pixmap(matrix=matrix,alpha=False)

            # Convertiamo la pagina PDF in array NumPy.
            image = np.frombuffer(pix.samples,dtype=np.uint8).reshape(pix.height,pix.width,pix.n)
            # OCR sulla pagina trasformata in immagine.
            results = reader.readtext(image,detail=0)
            testo_pagina = " ".join(results)
            testo_pagine.append(testo_pagina)
        document.close()

        # Uniamo il testo delle varie pagine.
        return "\n".join(testo_pagine)

    # ==========================================================
    # CASO 2: IMMAGINE
    # ==========================================================
    else:
        print("[INFO] Immagine rilevata.")
        results = reader.readtext(file_path,detail=0)
        return " ".join(results)


def clean_and_correct_text(raw_text):
    """
    Fase 2: PULIZIA (Pre-processing).
    Il testo grezzo dell'OCR contiene spesso errori (rumore). Qui cerchiamo di correggerli
    usando euristiche (RegEx) e similitudini probabilistiche.
    """
    print("[2] Post-processing (Pulizia e correzione)...")
    
    # A. Normalizzazione degli spazi bianchi.
    # Sostituisce sequenze multiple di spazi/tabulazioni con uno spazio singolo.
    text = re.sub(r'\s+', ' ', raw_text).strip()
    
    # B. Correzione di errori OCR specifici (Rule-Based).
    # Spesso l'OCR confonde la lettera 'O' maiuscola con il numero '0' o viceversa.
    # La regex r'(\d)[Oo]' cerca una cifra (\d) seguita da 'O' o 'o'.
    # r'\1 0' la sostituisce con la stessa cifra (\1) seguita da '0'.
    # Es: "2O26" (Due-O-Due-Sei) diventa "20 26" -> poi sistemato meglio o "2026" se adiacente.
    # Nota: la regex qui sotto sostituisce [Cifra][LetteraO] -> [Cifra][Spazio][Zero].
    text = re.sub(r'(\d)[Oo]', r'\1 0', text) 
    
    # C. Correzione Semantica (Fuzzy Logic).
    # Abbiamo un vocabolario di parole chiave che CI ASPETTIAMO di trovare.
    keywords_vocab = ["fattura", "importo", "totale", "scadenza", "cliente", "quantita" ,"consegna", "articolo"]
    candidate_labels = [
        "ordine di acquisto emesso da un cliente verso un fornitore",
        "fattura commerciale emessa da un fornitore per richiedere un pagamento",
        "richiesta di offerta o preventivo",
        "reclamo di un cliente",
        "documento tecnico"
    ]
    label_map = {
        "ordine di acquisto emesso da un cliente verso un fornitore": "Ordine",
        "fattura commerciale emessa da un fornitore per richiedere un pagamento": "Fattura",
        "richiesta di offerta o preventivo": "Richiesta offerta",
        "reclamo di un cliente": "Reclamo",
        "documento tecnico": "Documento Tecnico"
    }
    
    words = text.split()
    corrected = []
    
    for word in words:
        # Puliamo la parola da punteggiatura per il confronto
        clean = word.strip(".,:;").lower()
        
        # get_close_matches: Cerca nel vocabolario se esiste una parola molto simile (es. "fttura" -> "fattura").
        # - n=1: Prendi solo il match migliore.
        # - cutoff=0.8: La somiglianza deve essere almeno dell'80%.
        matches = get_close_matches(clean, keywords_vocab, n=1, cutoff=0.8)
        
        if matches:
            # Se troviamo un match, usiamo quello corretto.
            # Manteniamo la formattazione originale (MAIUSCOLO se l'originale era MAIUSCOLO).
            res = matches[0].upper() if word.isupper() else matches[0]
            corrected.append(res)
        else:
            # Se nessuna correzione è trovata, teniamo la parola originale.
            corrected.append(word)
            
    return " ".join(corrected)

def semantic_classification(cleaned_text):
    """
    Fase 3: classificazione zero-shot del tipo documento.
    """

    print("[3] Analisi NLP Zero-Shot...")

    device = 0 if torch.cuda.is_available() else -1

    model_name = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"

    classifier = pipeline(
        "zero-shot-classification",
        model=model_name,
        device=device
    )

    # Il documento BEPCO è prevalentemente francese.
    # Usiamo descrizioni semantiche complete invece di singole parole.
    candidate_labels = [
        "a purchase order sent by a customer to a supplier",
        "a commercial invoice sent by a supplier requesting payment",
        "a request for quotation or price",
        "a customer complaint",
        "a technical document"
    ]

    label_map = {
        "a purchase order sent by a customer to a supplier":
            "Ordine",

        "a commercial invoice sent by a supplier requesting payment":
            "Fattura",

        "a request for quotation or price":
            "Richiesta offerta",

        "a customer complaint":
            "Reclamo",

        "a technical document":
            "Documento Tecnico"
    }

    # Per identificare il TIPO di documento,
    # concentriamoci soprattutto sulla parte iniziale.
    classification_text = cleaned_text[:1800]

    result = classifier(
        classification_text,
        candidate_labels,
        hypothesis_template="This document is {}.",
        multi_label=False
    )

    print("\nPUNTEGGI ZERO-SHOT:")

    for label, score in zip(
        result["labels"],
        result["scores"]
    ):
        print(f"{label:<85} {score:.4f}")

    best_label = result["labels"][0]
    best_score = result["scores"][0]

    return {
        "categoria": label_map[best_label],
        "confidenza": float(best_score)
    }
# ==============================================================================
# 4. ESECUZIONE (MAIN LOOP)
# ==============================================================================
# Questo blocco viene eseguito solo se lanciamo lo script direttamente.

if __name__ == "__main__":
    # A. Gestione Dinamica dei Percorsi
    # __file__ è una variabile speciale che contiene il percorso di questo script.
    # Otteniamo la cartella genitore per assicurarci di leggere/scrivere nel posto giusto.
    #cartella_script = os.path.dirname(os.path.abspath(__file__))
    cartella_script = os.getcwd()
    print(f"Cartella script: {cartella_script}")
    nome_file = "example_complaint_letter.jpg"
    nome_file="BEPCO - ord_18732159.pdf"
    image_path = os.path.join(cartella_script, nome_file)
    
    # B. Generazione Dati di Test (se necessario)
    # Se l'immagine non esiste, la creiamo al volo con OpenCV.
    if not os.path.exists(image_path):
        print(f"Immagine non trovata. Creazione di {nome_file}...")
        # Creiamo un'immagine bianca (255) di dimensioni 250x900 pixel, con 3 canali colore (RGB).
        dummy_img = np.zeros((250, 900, 3), dtype=np.uint8) + 255
        
        # Scriviamo del testo simulato dentro l'immagine.
        # Nota l'errore intenzionale "2O26" (lettera O invece di zero) per testare la correzione.
        cv2.putText(dummy_img, "FATTURA N. 2O26/001", (50, 80), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0,0,0), 2)
        cv2.putText(dummy_img, "TOTALE DA PAGARE: 125,50 EURO", (50, 160), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0,0,0), 2)
        
        # Salviamo su disco.
        cv2.imwrite(image_path, dummy_img)

    # C. Esecuzione della Pipeline Sequenziale
    # 1. Image -> Text TESTO GREZZO
    raw_text = extract_text_simplified(image_path)
    print(raw_text)
    
    # 2. Dirty Text -> Clean Text TESTO PULITO
    clean_text = clean_and_correct_text(raw_text)
    
    # 3. Text -> Decision
    nlp_res = semantic_classification(clean_text)
    print(f"Decisione:\n{nlp_res}")

    # D. Output Formattato
    print("\n" + "="*60)
    print(" REPORT FINALE ELABORAZIONE DOCUMENTO ".center(60, "="))
    print("="*60)
    
    print(f"\n[ TESTO ESTRATTO (RAW) ]\n> {raw_text}")
    print(f"\n[ TESTO POST-PROCESSED (CLEAN) ]\n> {clean_text}")
    
    print("\n" + "-"*60)
    print(f"CATEGORIA STIMATA : {nlp_res['categoria'].upper()}")
    # Moltiplichiamo per 100 per avere la percentuale e tronchiamo a 2 decimali.
    print(f"GRADO DI CERTEZZA : {nlp_res['confidenza'] * 100:.2f}%")
    print("-"*60)
    print("="*60)

Using CPU. Note: This module is much faster with a GPU.



[STEP 0]: Inizializzazione EasyOCR...
Cartella script: c:\EPICODE\Epicode_Python_AI_MachineLearning\PYTHON\Modulo_5_ComputerVisione_NPL\06_ProgettiFinali

[1] Estrazione testo (OCR in corso)...
[INFO] Documento PDF rilevato.
[OCR] Elaborazione pagina 1/2...
[OCR] Elaborazione pagina 2/2...
bepco BEPCO FRANCE SHIP TO: 55 Allée de Martinon BEPCO FRANCE 5.A.S.- F - 47310, Sainte-Colombe-en-Bruilhois TECHNOPOLE AGEN GARONNE 55 ALLEE DE MAR 47310 SAINTE COLOMBE EN BRUILHOIS COMMANDE 18732159/31000 Date 28/7/2026 Numéro de fournisseur 00168492 DNP INDUSTRIALE SRL Veuillez mentionner_lors_de_la_confirmation_et facturatio Attn. Francesca CARZANIGA No. commande 18732159/31000 28/7/2026 Corso Magenta 56 Votre reference 20123 MILANO Monnaie EUR ITALIE TEL +39039877451 FAX E-MAIL sales@dnp.it Veuillez confirmer les prix et dates d'expédition. En cas d'une modification du prix ou du délai de livraison, attendez une confirmation. Pos.Réf Description Unitaire Date Sous-total Votre No. article Objet 

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 5813.47it/s]



PUNTEGGI ZERO-SHOT:
a request for quotation or price                                                      0.5551
a commercial invoice sent by a supplier requesting payment                            0.1779
a purchase order sent by a customer to a supplier                                     0.1390
a technical document                                                                  0.0873
a customer complaint                                                                  0.0407
Decisione:
{'categoria': 'Richiesta offerta', 'confidenza': 0.5551314353942871}

=========== REPORT FINALE ELABORAZIONE DOCUMENTO ===========

[ TESTO ESTRATTO (RAW) ]
> bepco BEPCO FRANCE SHIP TO: 55 Allée de Martinon BEPCO FRANCE 5.A.S.- F - 47310, Sainte-Colombe-en-Bruilhois TECHNOPOLE AGEN GARONNE 55 ALLEE DE MAR 47310 SAINTE COLOMBE EN BRUILHOIS COMMANDE 18732159/31000 Date 28/7/2026 Numéro de fournisseur 00168492 DNP INDUSTRIALE SRL Veuillez mentionner_lors_de_la_confirmation_et facturatio Attn. Frances